# 02 - Silver to Gold
Agregations metier (Batch Processing)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("02_Silver_to_Gold") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print("Spark connecte")

Spark connecte


In [2]:
# Lire les donnees Silver
trips = spark.read.parquet("hdfs://namenode:9000/projet/silver/trips_clean/")
dates = spark.read.parquet("hdfs://namenode:9000/projet/silver/dates_clean/")
locations = spark.read.parquet("hdfs://namenode:9000/projet/silver/locations_clean/")
weather = spark.read.parquet("hdfs://namenode:9000/projet/silver/weather_clean/")

print(f"Trips Silver: {trips.count()} lignes")

Trips Silver: 982149 lignes


In [3]:
# Agregation 1 : Stats par borough
stats_borough = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("borough")
    .agg(
        count("*").alias("nb_trajets"),
        round(avg("trip_distance"), 2).alias("distance_moyenne"),
        round(avg("total_amount"), 2).alias("tarif_moyen"),
        round(sum("total_amount"), 2).alias("revenu_total")
    )
    .orderBy(desc("nb_trajets"))
)
stats_borough.show()
stats_borough.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_borough/")

+-------------+----------+----------------+-----------+-------------+
|      borough|nb_trajets|distance_moyenne|tarif_moyen| revenu_total|
+-------------+----------+----------------+-----------+-------------+
|    Manhattan|    481490|            3.96|       24.0|1.155469944E7|
|        Bronx|    215641|             4.1|      30.61|   6601022.48|
|       Queens|    169709|            8.54|      41.15|   6984336.38|
|     Brooklyn|    103972|            6.69|      25.88|   2690591.08|
|Staten Island|     11303|            3.15|      27.67|    312797.92|
|          EWR|        34|            4.44|      98.84|      3360.42|
+-------------+----------+----------------+-----------+-------------+



In [4]:
# Agregation 2 : Stats par mois
stats_mensuel = (trips
    .join(broadcast(dates), "date_id")
    .groupBy("month", "year")
    .agg(
        count("*").alias("nb_trajets"),
        round(sum("total_amount"), 2).alias("revenu_total"),
        round(avg("trip_distance"), 2).alias("distance_moyenne")
    )
    .orderBy("year", "month")
)
stats_mensuel.show(12)
stats_mensuel.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_mois/")

+-----+----+----------+------------+----------------+
|month|year|nb_trajets|revenu_total|distance_moyenne|
+-----+----+----------+------------+----------------+
|    1|2024|     71131|  1946230.13|            3.77|
|    2|2024|     71928|  1960845.74|            3.22|
|    3|2024|     85046|  2376723.92|            3.38|
|    4|2024|     84724|  2388945.63|           10.27|
|    5|2024|     89279|  2586716.69|            6.91|
|    6|2024|     84509|  2432403.95|            5.55|
|    7|2024|     73441|   2134096.6|            3.62|
|    8|2024|     70798|  2061388.09|            5.39|
|    9|2024|     86004|  2531108.72|            5.65|
|   10|2024|     91151|  2682235.81|            4.07|
|   11|2024|     86975|  2480691.64|            3.97|
|   12|2024|     87162|   2565411.4|            4.37|
+-----+----+----------+------------+----------------+



In [5]:
# Agregation 3 : Top 10 zones
top_zones = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("zone", "borough")
    .agg(count("*").alias("nb_trajets"))
    .orderBy(desc("nb_trajets"))
    .limit(10)
)
top_zones.show(truncate=False)
top_zones.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/top_zones/")

+---------------------------------+---------+----------+
|zone                             |borough  |nb_trajets|
+---------------------------------+---------+----------+
|Melrose South                    |Bronx    |46117     |
|University Heights/Morris Heights|Bronx    |46041     |
|Jamaica                          |Queens   |46012     |
|Union Sq                         |Manhattan|41662     |
|Middle Village                   |Queens   |34169     |
|Sunset Park West                 |Brooklyn |33007     |
|Pelham Bay Park                  |Bronx    |32867     |
|Lenox Hill East                  |Manhattan|31912     |
|Kingsbridge Heights              |Bronx    |31286     |
|Mott Haven/Port Morris           |Bronx    |28155     |
+---------------------------------+---------+----------+



In [6]:
# Agregation 4 : Impact meteo
stats_meteo = (trips
    .filter(col("weather_id") != -1)
    .join(broadcast(weather), "weather_id")
    .groupBy("weather_id", "temp_max", "precipitation", "snow")
    .agg(
        count("*").alias("nb_trajets"),
        round(avg("trip_distance"), 2).alias("distance_moyenne"),
        round(avg("total_amount"), 2).alias("tarif_moyen")
    )
)
stats_meteo.show()
stats_meteo.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_meteo/")

+----------+--------+-------------+----+----------+----------------+-----------+
|weather_id|temp_max|precipitation|snow|nb_trajets|distance_moyenne|tarif_moyen|
+----------+--------+-------------+----+----------+----------------+-----------+
|      19.0|   29.33|         0.09| 0.0|      2623|            3.68|      29.85|
|      30.0|   21.45|         0.02| 0.0|      2876|            3.24|      28.62|
|      28.0|   27.15|         0.76| 0.0|      1869|            4.95|      32.87|
|      18.0|   28.45|          0.0| 0.0|      2551|            3.98|      30.48|
|      26.0|    33.6|        15.68| 0.0|      2079|            4.02|      29.57|
|       2.0|    6.53|         0.03| 0.0|      1826|            4.29|      31.58|
|      38.0|    5.33|         0.02| 0.0|      3214|            3.45|      30.53|
|       6.0|    6.12|         0.95| 0.0|      2612|            2.77|      24.49|
|      23.0|    33.7|         0.53| 0.0|      2763|            3.31|      28.29|
|      20.0|   28.37|       

In [7]:
# Agregation 5 : Matrice Origine-Destination (Top 20)
loc_pu = locations.withColumnRenamed("zone", "zone_pickup") \
    .withColumnRenamed("location_id", "pickup_location_id") \
    .select("pickup_location_id", "zone_pickup")

loc_do = locations.withColumnRenamed("zone", "zone_dropoff") \
    .withColumnRenamed("location_id", "dropoff_location_id") \
    .select("dropoff_location_id", "zone_dropoff")

od_matrix = (trips
    .join(broadcast(loc_pu), on="pickup_location_id")
    .join(broadcast(loc_do), on="dropoff_location_id")
    .groupBy("zone_pickup", "zone_dropoff")
    .agg(
        count("*").alias("nb_trajets"),
        round(avg("total_amount"), 2).alias("tarif_moyen")
    )
    .orderBy(desc("nb_trajets"))
    .limit(20)
)
od_matrix.show(truncate=False)
od_matrix.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/od_matrix/")

print("Silver -> Gold TERMINE")

+---------------------------------+---------------------------------+----------+-----------+
|zone_pickup                      |zone_dropoff                     |nb_trajets|tarif_moyen|
+---------------------------------+---------------------------------+----------+-----------+
|University Heights/Morris Heights|Union Sq                         |6739      |15.68      |
|Union Sq                         |University Heights/Morris Heights|5864      |16.16      |
|University Heights/Morris Heights|University Heights/Morris Heights|4610      |13.88      |
|Union Sq                         |Union Sq                         |4514      |13.26      |
|Melrose South                    |University Heights/Morris Heights|3200      |17.5       |
|University Heights/Morris Heights|Melrose South                    |2798      |17.91      |
|Melrose South                    |Union Sq                         |2664      |22.26      |
|Lenox Hill East                  |Upper East Side South            |2

In [8]:
spark.stop()